In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["churn"] = (df["Churn"] == "Yes").astype(int)

# Colonnes écartées : l'identifiant, la cible, et les données sensibles
# exclues par choix éthique (cf. docs/architecture.md)
dropped = ["customerID", "gender", "SeniorCitizen", "Partner", "Dependents", "Churn"]
X = df.drop(columns=dropped + ["churn"])
y = df["churn"]

print(X.shape, "| taux de départ :", y.mean().round(3))
X.columns.tolist()

(7043, 15) | taux de départ : 0.265


['tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges']

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(len(X_train), len(X_test), "|", y_train.mean().round(3), y_test.mean().round(3))

5634 1409 | 0.265 0.265


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_cols = [c for c in X.columns if c not in numeric_cols]

preprocess = ColumnTransformer(
    [
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value=0)),
            ("scale", StandardScaler()),
        ]), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ],
    sparse_threshold=0,
)

In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, recall_score, roc_auc_score


def evaluate(name, model):
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {
        "modèle": name,
        "AUC": roc_auc_score(y_test, proba),
        "AUC-PR": average_precision_score(y_test, proba),
        "rappel": recall_score(y_test, pred),
        "précision": precision_score(y_test, pred, zero_division=0),
    }


results = [
    evaluate("Référence (toujours « reste »)",
             Pipeline([("prep", preprocess), ("clf", DummyClassifier(strategy="prior"))])),
    evaluate("Régression logistique",
             Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=1000))])),
]
pd.DataFrame(results).round(3)

,modèle,AUC,AUC-PR,rappel,précision
0,Référence (toujours « reste »),0.500,0.265,0.000,0.000
1,Régression logistique,0.841,0.634,0.545,0.638


In [5]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

candidates = {
    "Régression logistique": LogisticRegression(max_iter=1000),
    "Forêt aléatoire": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1),
    "Gradient boosting": HistGradientBoostingClassifier(
        max_depth=3, learning_rate=0.05, max_iter=200, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for name, clf in candidates.items():
    pipe = Pipeline([("prep", preprocess), ("clf", clf)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv,
                            scoring=["roc_auc", "average_precision"])
    rows.append({
        "modèle": name,
        "AUC (moyenne)": scores["test_roc_auc"].mean(),
        "AUC (écart)": scores["test_roc_auc"].std(),
        "AUC-PR (moyenne)": scores["test_average_precision"].mean(),
        "AUC-PR (écart)": scores["test_average_precision"].std(),
    })

pd.DataFrame(rows).round(3)

,modèle,AUC (moyenne),AUC (écart),AUC-PR (moyenne),AUC-PR (écart)
0,Régression logistique,0.846,0.013,0.660,0.018
1,Forêt aléatoire,0.844,0.010,0.660,0.021
2,Gradient boosting,0.847,0.011,0.668,0.021
